In [68]:
import pandas as pd
from umi_tools import UMIClusterer


def cluster_umis(moleculetable, cluster_method = "directional", threshold = 1):
    df.set_index('umi_seq', inplace = True)
    #get gavage columns, create sum value and :
    df_gavage = df[[col for col in df.columns if 'gavage' in str(col).lower()]].copy()
    dict_gavage = df_gavage.sum(axis=1).to_dict()

    # Encode keys for umi_tools cluster
    umi_dict = {key.encode(): value for key, value in dict_gavage.items()}
    clusterer = UMIClusterer(cluster_method=cluster_method)
    clustered_umis = clusterer(umi_dict, threshold=threshold)
    li_clusters = [len(cluster) for cluster in clustered_umis]
    sum_clusters = sum(li_clusters)
    print('clustered '+ str(len(clustered_umis)) + ' of '+ str(sum_clusters) + ' total umis')

    # Build a list of Series, then concat once at the end
    cluster_series_list = []

    for cluster in clustered_umis:
        # Get representative UMI (first in cluster)
        representative_umi = cluster[0].decode()
        
        # Get all UMIs in this cluster
        cluster_umis = [umi.decode() for umi in cluster]
        
        # Get all rows from original dataframe for UMIs in this cluster
        cluster_rows = df.loc[cluster_umis]
        
        # Sum ALL columns across these rows
        summed_row = cluster_rows.sum(axis=0)
        summed_row.name = representative_umi  # Set the index name
        
        # Append to list instead of adding to DataFrame
        cluster_series_list.append(summed_row)

    # Concat all at once - much faster!
    clustered_df = pd.concat(cluster_series_list, axis=1).T
    clustered_df.reset_index(inplace = True)
    clustered_df = clustered_df.rename(columns = {'index': 'umi_seq'})

    return clustered_df

In [ ]:
df = pd.read_csv('/Volumes/sd/faith/MTCSB/projects/P4-barcoding_strains/20251031_analysis/fp_inputs/P4C1T8-c1gavage/c1_ST1_onlygavage_moleculestable.csv')
df = df.head(1000).copy()
df_cluster = cluster_umis(df, cluster_method = "directional", threshold = 1)



clustered 604 of 1000 total umis


,umi_seq,s1_gavage-DNA,s100_SI-DNA,s109_Cecum-DNA,s110_Cecum-DNA,s119_Pr.-Colon-DNA,s120_Pr.-Colon-DNA,s19_stool-DNA,s20_stool-DNA,s29_stool-DNA,...,s39_stool-DNA,s40_stool-DNA,s49_stool-DNA,s50_stool-DNA,s99_SI-DNA,NTC_H3_NTC,WATER_B3_WATER,WATER_C6_WATER,WATER_D2_WATER,WATER_F9_WATER
0,ST1-AAAAGACTGGGCCTTTCG,987688,10,6249595,1407881,4871,7035,1368262,1077,1301080,...,4707200,141979,1064753,409995,14321,0,3,35,126,4
1,ST1-AAACAAAATACAGCATCC,2152,0,167,0,15,0,0,0,1700,...,143,0,3681,0,0,0,0,0,0,0
2,ST1-AAACAAACTGCGATATAG,1720,0,4750,0,0,0,83,0,2080,...,10571,0,139,333,0,0,0,0,0,0
3,ST1-AAACAAAACGCCACAAGG,1624,0,6875,0,7,0,208,0,120,...,2786,0,1444,0,0,0,0,0,0,0
4,ST1-AAACAAAACACCCCACTA,1616,1334,583,222,0,12,0,0,2520,...,25571,13738,14,0,158,0,0,0,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,ST1-AAACAAAGCGACCCGATT,8,0,42,0,0,0,0,0,3300,...,286,0,0,0,0,0,0,0,0,0
600,ST1-AAACAAAGCGGAACGGCT,8,0,5625,0,0,0,10208,0,3720,...,714,0,42,0,0,0,0,0,0,0
601,ST1-AAACAAAGTAAATTGAAA,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
602,ST1-AAACAAAGTAATGCATGT,8,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
